In [1]:
!pip install "qiskit[visualization]>=2.0" "qiskit-ibm-runtime>=0.40"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.6/412.6 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.7/120.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.3/113.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.2/234.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.6/76.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.3/130.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.5/54

In [1]:
import qiskit, qiskit_ibm_runtime
print(qiskit.__version__, qiskit_ibm_runtime.__version__)

2.5.2 0.50.0


In [2]:
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler

qc = QuantumCircuit(2)
qc.h(0)       # 讓 q0 進入疊加態
qc.cx(0, 1)   # 讓 q1 和 q0 糾纏
qc.measure_all()
print(qc.draw())

result = StatevectorSampler().run([qc], shots=1000).result()
print(result[0].data.meas.get_counts())

        ┌───┐      ░ ┌─┐   
   q_0: ┤ H ├──■───░─┤M├───
        └───┘┌─┴─┐ ░ └╥┘┌─┐
   q_1: ─────┤ X ├─░──╫─┤M├
             └───┘ ░  ║ └╥┘
meas: 2/══════════════╩══╩═
                      0  1 
{'11': 501, '00': 499}


In [3]:
import math
from qiskit.circuit.library import grover_operator, MCMTGate, ZGate

def grover_oracle(target):
    n = len(target)
    qc = QuantumCircuit(n)
    # Qiskit 的字串最右邊是 q0，所以要反轉
    zero_idx = [i for i, bit in enumerate(reversed(target)) if bit == "0"]
    if zero_idx:
        qc.x(zero_idx)           # 把目標裡的 0 暫時翻成 1
    qc.compose(MCMTGate(ZGate(), n - 1, 1), inplace=True)  # 全部是 1 時翻相位
    if zero_idx:
        qc.x(zero_idx)           # 翻回來
    return qc

oracle = grover_oracle("101")
print(oracle.draw())

           1-Z      
q_0: ───────■───────
     ┌───┐  │  ┌───┐
q_1: ┤ X ├──■──┤ X ├
     └───┘  │  └───┘
q_2: ───────■───────
                    


In [4]:
def build_grover(target, k):
    oracle = grover_oracle(target)
    grover_op = grover_operator(oracle)   # Oracle + Diffuser
    n = grover_op.num_qubits
    qc = QuantumCircuit(n)
    qc.h(range(n))                        # 所有狀態平均疊加
    qc.compose(grover_op.power(k), inplace=True)  # 做 k 次迭代
    qc.measure_all()
    return qc

target = "101"
shots = 4000
for k in [1, 2, 3]:
    qc = build_grover(target, k)
    counts = StatevectorSampler().run([qc], shots=shots).result()[0].data.meas.get_counts()
    print(k, counts.get(target, 0) / shots)

1 0.7835
2 0.9435
3 0.3275


In [13]:
for t, k in [("101", 1), ("101", 2), ("101", 3)]:
    qc = build_grover(t, k)
    counts = StatevectorSampler().run([qc], shots=4000).result()[0].data.meas.get_counts()
    print(t, k, counts.get(t, 0) / 4000)

101 1 0.79175
101 2 0.9505
101 3 0.32275


In [ ]:
from qiskit.visualization import plot_histogram
qc = build_grover(target, 2)
counts = StatevectorSampler().run([qc], shots=shots).result()[0].data.meas.get_counts()
plot_histogram(counts, title="Simulator, k=2").savefig("sim_k2.png", dpi=200, bbox_inches="tight")

In [9]:
from getpass import getpass
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(token=getpass("API key: "), instance="crn:v1:bluemix:public:quantum-computing:us-east:a/3fc52cbaa6784f9cb83f6cbc649ef866:4011e7a8-b7d8-42f4-bd26-5b5d76d2c23a::")
backend = service.least_busy(operational=True, simulator=False)
print(backend.name, backend.num_qubits)

API key: ··········


qiskit_runtime_service._discover_account:WARNING:2026-09-30 08:17:03,021: Loading account with the given token. A saved account will not be used.


ibm_kingston 156


In [10]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler

pm = generate_preset_pass_manager(target=backend.target, optimization_level=3)
experiments = [("101", 1), ("101", 2), ("1011", 3)]   # 3 位元 k=1、k=2，4 位元 k=3
circuits = [pm.run(build_grover(t, k)) for t, k in experiments]

for (t, k), c in zip(experiments, circuits):
    print(t, k, "深度", c.depth(), "雙位元閘", c.num_nonlocal_gates())

sampler = Sampler(mode=backend)
job = sampler.run(circuits, shots=4000)
print(job.job_id())   # 記下來，排隊太久可以之後再取結果

101 1 深度 72 雙位元閘 19
101 2 深度 139 雙位元閘 39
1011 3 深度 510 雙位元閘 161


/tmp/ipykernel_896/3735458572.py:11: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  sampler = Sampler(mode=backend)


daucbarg95ks73ei57og


In [14]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler

pm = generate_preset_pass_manager(target=backend.target, optimization_level=3)
experiments = [("101", 1), ("101", 2), ("101", 3)]   # 3 位元 k=1、k=2，3 位元 k=3
circuits = [pm.run(build_grover(t, k)) for t, k in experiments]

for (t, k), c in zip(experiments, circuits):
    print(t, k, "深度", c.depth(), "雙位元閘", c.num_nonlocal_gates())

sampler = Sampler(mode=backend)
job = sampler.run(circuits, shots=4000)
print(job.job_id())   # 記下來，排隊太久可以之後再取結果

101 1 深度 72 雙位元閘 19
101 2 深度 134 雙位元閘 37
101 3 深度 201 雙位元閘 57


/tmp/ipykernel_896/4180385712.py:11: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  sampler = Sampler(mode=backend)


daucij2hcrkc73dvdrsg


In [11]:
job = service.job("daucbarg95ks73ei57og")
res = job.result()
for i, (t, k) in enumerate(experiments):
    counts = res[i].data.meas.get_counts()
    print(t, k, counts.get(t, 0) / 4000)

101 1 0.69725
101 2 0.8
1011 3 0.58575


In [15]:
job = service.job("daucij2hcrkc73dvdrsg")
res = job.result()
for i, (t, k) in enumerate(experiments):
    counts = res[i].data.meas.get_counts()
    print(t, k, counts.get(t, 0) / 4000)

101 1 0.621
101 2 0.66
101 3 0.25525
